# 🏥 Healthcare: Readmission Risk Prediction (Kaggle Style)

LightGBM model with SHAP and Gradio UI.

In [ ]:
!pip install lightgbm shap gradio --quiet

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import shap
import gradio as gr


In [ ]:
np.random.seed(1212)
n = 1000
df = pd.DataFrame({
    "feature1": np.random.rand(n),
    "feature2": np.random.randn(n),
    "feature3": np.random.randint(0, 2, n),
    "target": np.random.choice([0, 1], n, p=[0.7, 0.3])
})


In [ ]:
X = df.drop(columns="target")
y = df["target"]
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.2)
model = lgb.LGBMClassifier()
model.fit(X_train, y_train)
print("AUC:", roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))


In [ ]:
explainer = shap.Explainer(model)
shap_values = explainer(X_test)
shap.summary_plot(shap_values, X_test)


In [ ]:
def predict(f1, f2, f3):
    row = pd.DataFrame([[f1, f2, f3]], columns=["feature1", "feature2", "feature3"])
    prob = model.predict_proba(row)[0][1]
    return f"Prediction Score: {prob:.2%}"

gr.Interface(
    fn=predict,
    inputs=[gr.Number(label="Feature 1"), gr.Number(label="Feature 2"), gr.Radio([0, 1], label="Feature 3")],
    outputs="text",
    title="Healthcare AI Demo"
).launch()
